# AlphaZero Chess — Google Colab Training

**Prerequisites:**
- Runtime → Change runtime type → **T4 GPU** (or better)
- Have your GitHub repo URL ready

**Workflow:**
1. Run cells 1–4 **once** per session to set up the environment
2. Run cell 5 to start or resume training
3. If the session disconnects, re-run cells 1 + 3 + 5 (Drive stays mounted between reconnects)
4. Run cell 6 at any time to download `best.pth.tar` locally

## Cell 1 — Clone repo & install dependencies
*(Run once per session)*

In [ ]:
# ── Edit this URL to point to your GitHub repo ────────────────────────────
REPO_URL = 'https://github.com/minghanminghan/chess-bot.git'
REPO_DIR = 'chess-bot'   # local clone directory name
# ─────────────────────────────────────────────────────────────────────────

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'{REPO_DIR}/ already exists — skipping clone')

%cd {REPO_DIR}

# Install PyTorch with CUDA 12.1 (matches Colab's current CUDA)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
# Other deps
!pip install python-chess numpy tqdm -q

print('\nSetup complete.')

## Cell 2 — Mount Google Drive
*(Run once per session — checkpoints survive disconnects here)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/chess-bot-checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')
print(f'Existing files: {os.listdir(CKPT_DIR) or "(empty)"}')

## Cell 3 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No GPU detected. Go to Runtime → Change runtime type → Hardware accelerator → GPU'
)
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')
print(f'RAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 4 — Pull latest code from GitHub
*(Run whenever you push local changes)*

In [ ]:
!git pull origin main

## Cell 5 — Run training
*(Re-run this cell after any session disconnect — set `RESUME = True` after the first run)*

In [ ]:
RESUME = True   # False = fresh run;  True = continue from best.pth.tar

resume_flag = '--resume' if RESUME else ''

!python train.py \
  --checkpoint-dir "{CKPT_DIR}" \
  {resume_flag}

## Cell 6 — Download best checkpoint to local machine

In [ ]:
import os
from google.colab import files

ckpt_path = os.path.join(CKPT_DIR, 'best.pth.tar')
if os.path.isfile(ckpt_path):
    files.download(ckpt_path)
    print('Download started.')
else:
    print(f'No checkpoint found at {ckpt_path}. Train first.')

## Cell 7 — Scale-down test run (optional)
Verifies the full pipeline end-to-end in ~10 minutes using a tiny model.
Use this to sanity-check before committing to a full training run.

In [ ]:
TEST_CKPT_DIR = CKPT_DIR + '/test-run'

!python train.py \
  --checkpoint-dir "{TEST_CKPT_DIR}" \
  --num-iters 5 \
  --num-eps 5 \
  --mcts-sims 25 \
  --num-channels 64 \
  --num-res-blocks 4